In [36]:
import pandas as pd 
import numpy as np 
from sklearn.linear_model import Lasso
from scipy.stats import norm
import os

In [37]:
data_path = os.path.join("Data","acemoglu64.csv")
dat = pd.read_csv(data_path)
print(f'The data contains {dat.shape[0]} rows (countries) and {dat.shape[1]} columns (variables).')

The data contains 64 rows (countries) and 51 columns (variables).


In [38]:
Y = dat['logpgp95']
A = dat['avexpr']
Z = dat['logem4']
X = dat.drop(columns=['logpgp95','avexpr','logem4', 'shortnam'])

In [39]:
def standardize(X):
    X = np.asarray(X, dtype=float)
    mu = X.mean(axis=0, keepdims=True)
    sd = X.std(axis=0, ddof=1, keepdims=True)
    sd[sd == 0] = 1.0
    return (X - mu) / sd

def BCCH(X_tilde, y, c=1.1, alpha=0.05):
    X_tilde = np.asarray(X_tilde, dtype=float)
    y = np.asarray(y, dtype=float).flatten()
    N, p = X_tilde.shape    

    yXscale = (np.max((X_tilde.T ** 2) @ ((y-np.mean(y)) ** 2) / N)) ** 0.5
    lambda_pilot = c*norm.ppf(1-alpha/(2*p))*yXscale/np.sqrt(N)
    
    pred = Lasso(alpha=lambda_pilot).fit(X_tilde,y).predict(X_tilde)

    res = y - pred
    resXscale = (np.max((X_tilde.T ** 2) @ (res ** 2) / N)) ** 0.5
    lambda_bcch = c*norm.ppf(1-alpha/(2*p))*resXscale/np.sqrt(N)

    return lambda_bcch

In [40]:
mask = np.isfinite(Y) & np.isfinite(X).all(axis=1)

# 2. Count how many remain
N_before = X.shape[0]
N_after = mask.sum()
removed = N_before - N_after

print(f"Original observations: {N_before}")
print(f"Remaining after dropping rows with NaNs: {N_after}")
print(f"Dropped observations: {removed}")

# 3. Apply mask
X = X[mask]
Y = Y[mask]
A = A[mask]
Z = Z[mask]

Original observations: 64
Remaining after dropping rows with NaNs: 57
Dropped observations: 7


In [41]:
X_tilde = standardize(X)


In [ ]:
print("X_tilde shape:", X_tilde.shape)
print("Any NaN in X_tilde?", np.isnan(X_tilde).any())
print("Std of columns (should not be ~0):")
print(np.std(X_tilde, axis=0)[:20])  # first 20


X_tilde shape: (57, 47)
Any NaN in X_tilde? False
Any NaN in Y_clean? False
Std of columns (should not be ~0):
[0.99118926 0.99118926 0.99118926 0.99118926 0.99118926 0.99118926
 0.99118926 0.99118926 0.99118926 0.99118926 0.99118926 0.99118926
 0.99118926 0.99118926 0.99118926 0.99118926 0.99118926 0.99118926
 0.99118926 0.99118926]


In [ ]:
# 1st LASSO: Y on X
lambda_Y = BCCH(X_tilde, Y)
lasso_Y = Lasso(alpha=lambda_Y, max_iter=12000).fit(X_tilde, Y)
coefs_Y = lasso_Y.coef_
sel_Y = np.where(coefs_Y != 0)[0]
# 2nd LASSO: A on X
lambda_A = BCCH(X_tilde, A) 
lasso_A = Lasso(alpha=lambda_A, max_iter=12000).fit(X_tilde, A)
coefs_A = lasso_A.coef_
sel_A = np.where(coefs_A != 0)[0]
# 3rd LASSO: Z on X
lambda_Z = BCCH(X_tilde, Z)
lasso_Z = Lasso(alpha=lambda_Z, max_iter=12000).fit(X_tilde, Z)
coefs_Z = lasso_Z.coef_
sel_Z = np.where(coefs_Z != 0)[0]
# Union of selected variables
sel_union = np.union1d(sel_Y, np.union1d(sel_A, sel_Z))
print(f'Selected variables indices: {sel_union}')

Selected variables indices: []


In [49]:
from sklearn.linear_model import Lasso
from scipy.stats import norm
import numpy as np

def BCCH_diag(X_tilde, y, c=1.1, alpha=0.05, max_iter=10000):
    # Make sure inputs are nice
    X_tilde = np.asarray(X_tilde, dtype=float)
    y = np.asarray(y, dtype=float).reshape(-1)
    N, p = X_tilde.shape

    if p == 0:
        raise ValueError("X_tilde must have at least one column.")
    if N != y.shape[0]:
        raise ValueError(f"Row mismatch: X_tilde has {N} rows, y has {y.shape[0]}.")

    if not np.isfinite(X_tilde).all():
        raise ValueError("X_tilde contains NaN or inf.")
    if not np.isfinite(y).all():
        raise ValueError("y contains NaN or inf.")

    # Quantile term
    quant = norm.ppf(1 - alpha / (2 * p))

    # Pilot scale based on y
    y_centered = y - y.mean()
    yXscale = np.sqrt(np.max((X_tilde ** 2).T @ (y_centered ** 2) / N))
    lambda_pilot = c * quant * yXscale / np.sqrt(N)

    print("=== BCCH diagnostics ===")
    print(f"N             = {N}")
    print(f"p             = {p}")
    print(f"quantile term = {quant:.6f}")
    print(f"yXscale       = {yXscale:.6f}")
    print(f"lambda_pilot  = {lambda_pilot:.6f}")

    # Pilot Lasso
    fit_pilot = Lasso(alpha=lambda_pilot, max_iter=max_iter).fit(X_tilde, y)
    eps = y - fit_pilot.predict(X_tilde)

    # Final scale based on residuals
    epsXscale = np.sqrt(np.max((X_tilde ** 2).T @ (eps ** 2) / N))
    lambda_bcch = c * quant * epsXscale / np.sqrt(N)

    print(f"epsXscale     = {epsXscale:.6f}")
    print(f"lambda_bcch   = {lambda_bcch:.6f}")
    print("=========================\n")

    return float(lambda_bcch)
# Compute BCCH penalty with diagnostics
lambda_Y = BCCH_diag(X_tilde, Y)

# Fit Lasso and see selection
lasso_Y = Lasso(alpha=lambda_Y, max_iter=12000).fit(X_tilde, Y)
coefs_Y = lasso_Y.coef_
selected = np.where(coefs_Y != 0)[0]

print("BCCH lambda_Y:", lambda_Y)
print("Number of selected variables:", len(selected))
print("Selected indices:", selected)
for f in [1.0, 0.8, 0.2, 0.1]:
    lmb = lambda_Y * f
    lasso_tmp = Lasso(alpha=lmb, max_iter=12000).fit(X_tilde, Y)
    k_sel = np.sum(lasso_tmp.coef_ != 0)
    print(f"alpha = {lmb:.6f}, selected = {k_sel}")


=== BCCH diagnostics ===
N             = 57
p             = 47
quantile term = 3.273078
yXscale       = 1.847714
lambda_pilot  = 0.881143
epsXscale     = 1.847714
lambda_bcch   = 0.881143

BCCH lambda_Y: 0.8811430348748962
Number of selected variables: 0
Selected indices: []
alpha = 0.881143, selected = 0
alpha = 0.704914, selected = 1
alpha = 0.176229, selected = 6
alpha = 0.088114, selected = 9
